In [1]:
import os
import cv2
import numpy as np
import trimesh

from to3d import map_to_pixel, sample_image, crop_img, divide

In [ ]:
cloth_path = 'deform-inpaint-037.ply'
tex_num = 4
texture_path = f'texture_diffuse{tex_num}.png'
save_path = 'outputs'

cloth_final = trimesh.load(cloth_path, validate=False, process=False)
cloth_final.vertices /= 0.8

pattern_f = trimesh.load('pattern-f-037.ply', validate=False, process=False)
pattern_b = trimesh.load('pattern-b-037.ply', validate=False, process=False)

_max = max(pattern_f.vertices.max(), -pattern_f.vertices.min())
scale = 1./(_max)

texture = cv2.imread(texture_path)[:,:,::-1].copy()
texture = cv2.rotate(texture, cv2.ROTATE_90_CLOCKWISE)
texture = crop_img(texture)

# contour_f = get_contour(texture, pattern_f, scale)
# contour_b = get_contour(texture, pattern_b, scale)
# cv2.imwrite(os.path.join(save_path, 'texture.png'), texture[:,:,::-1])
# cv2.imwrite(os.path.join(save_path, 'contour_f_skirt.png'), contour_f)
# cv2.imwrite(os.path.join(save_path, 'contour_b_skirt.png'), contour_b)

pattern_v = np.concatenate((pattern_f.vertices, pattern_b.vertices), axis=0)
pattern_f = np.concatenate((pattern_f.faces, len(pattern_f.vertices)+pattern_b.faces), axis=0)
pattern = trimesh.Trimesh(pattern_v, pattern_f, validate=False, process=False)
cloth_pose_sub, pattern_v_new = divide(cloth_final, pattern)

pos = map_to_pixel(pattern_v_new, texture, scale=scale)

color = sample_image(texture, pos.T)
cloth_pose_sub.visual.vertex_colors = color.astype(np.uint8)


out_path = os.path.join(save_path, os.path.basename(cloth_path).split('.')[0] + f'-tex{tex_num}.ply')
cloth_pose_sub.export(out_path)

In [ ]:
# Load the mesh
out_path = 'outputs/deform-inpaint-037-tex1.ply'
mesh = trimesh.load(out_path)

# Show in viewer
mesh.show()